In [31]:
import sys
import glob
import pickle
import numpy as np
import pandas as pd
from functools import reduce

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")

import Chronocell

In [32]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.5_208_genes_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [83]:
t = traj.t

In [58]:
# Y = traj.X
Q = traj.Q[:, 0, :] 
# tau = traj.tau # State transition times (global)
#  = traj.t
# theta = traj.theta
# topo = traj.topo
# state_grid = np.searchsorted(tau, t, side="left") - 1
# state_grid[0] = 0 

## Get genes of interest

In [3]:
genes = pd.read_csv("eLNPs_var>1.5_208_genes.csv")
shared_genes = pd.read_csv("RNA_vs_ADT_corr_meanExpr.csv")
gene_idx = genes['Gene_symbol'].isin(shared_genes['Gene']).tolist()

In [4]:
genes.iloc[gene_idx]['Gene_symbol'].tolist()

['Cxcr4',
 'Icos',
 'Cd48',
 'Sbno2',
 'Cd24a',
 'Cd63',
 'Cd68',
 'Ccr7',
 'Havcr2',
 'Ighm',
 'Cd83',
 'Lgals3',
 'Il7r',
 'Ly6a',
 'Ly6e',
 'Cd86',
 'Cd200',
 'Cd80',
 'Btla',
 'Cd274',
 'Fas',
 'Dpp4',
 'Il2ra',
 'Itga4',
 'Procr',
 'Cd1d1',
 'Vcam1',
 'Tnfrsf4',
 'Tnfrsf18',
 'Cd36',
 'Sbno1',
 'Flt3',
 'Cd38',
 'Kit',
 'Cd9',
 'Cd69',
 'Klrk1',
 'Cd8a',
 'Clec12a',
 'Clec9a',
 'Pirb',
 'Ilk',
 'Il4ra',
 'Itgal',
 'Itgax',
 'Bst2',
 'Lamp1',
 'Itgb1',
 'Xcr1',
 'Icam1',
 'Jaml']

## Get half-life data

In [5]:
data_dir = "/mnt/lareaulab/reliscu/projects/Chronocell/data/experimental_parameters/protein_degradation_rates"
file_list = glob.glob(f"{data_dir}/*standardized.csv")
subset_columns = ['Gene', 'Half-life']

df_list = []
for file in [file for file in file_list if "Mouse" in file]:
    df = pd.read_csv(file)
    study = df['Study'].iloc[0]
    cell_type = df['Cell_type'].iloc[0]
    df = df[subset_columns]
    df = df.rename(columns={"Half-life": f"Half-life_{study}_{cell_type}", 
                            "Cell_type": f"Cell_type_{study}"}) 
    df_list.append(df)

In [6]:
df_list[0].head()

,Gene,Half-life_McShane2016_NIH3T3 fibroblasts
0,Cul4b,31.378265
1,Dhx8,28.700000
2,Znf335,3.599724
3,Arfgef2,24.371302
4,Cdc27,21.057713


In [7]:
df = reduce(lambda x, y: pd.merge(x, y, on="Gene", how="outer"), df_list)


## Subset half-life data to genes of interest

In [8]:
df_subset = df.loc[df['Gene'].isin(genes.iloc[gene_idx]['Gene_symbol'])]

## Convert to degradation rate

In [9]:
half_life_cols = [col for col in df_subset.columns if "Half-life" in col]
df_deg = np.log(2)/df_subset[half_life_cols]
median_deg = np.nanmedian(df_deg, axis=1)

In [10]:
df_subset['Gene']

1962       Bst2
2973       Cd63
2976        Cd9
7581        Ilk
7683      Itgb1
8793      Lamp1
8875     Lgals3
12968     Procr
16288     Sbno1
16289     Sbno2
18975     Vcam1
Name: Gene, dtype: object

## Impute protein

In [ ]:

####

i = 0 # Gene index

gene = df_subset['Gene'].iloc[0]
file_path = f"data/RNA_history_per_gene/eLNPs_var>1.5_208_genes_{gene}_S_RNA_history.pkl"

with open(file_path, "rb") as f:
    X_bw = pickle.load(f)
    
deg_rate = median_deg[0]
k_trans = 1


#######



In [254]:
df_subset['Gene']

1962       Bst2
2973       Cd63
2976        Cd9
7581        Ilk
7683      Itgb1
8793      Lamp1
8875     Lgals3
12968     Procr
16288     Sbno1
16289     Sbno2
18975     Vcam1
Name: Gene, dtype: object

In [250]:
def impute_protein(X_bw, Q, t, deg_rate, transl_rate=1):
    Q_max_idx = np.argmax(Q, axis=1) # Each cell's time along the trajectory

    y0 = X_bw[:, 0] # Steady-state RNA abundance per gene
    ss_rate = transl_rate / deg_rate # Steady-state protein production rate 
    p0 = ss_rate * y0 # Steady-state protein abundance
    p_old = p0 * np.exp(-deg_rate * t[Q_max_idx]) # Pre-existing protein that has not yet degraded at the time each cell was observed

    t_reshape = t.reshape((-1, 1))
    t_diff = t_reshape.T - t_reshape
    t_diff = t_diff[:, Q_max_idx] # Each column corresponds to a cell: contains time steps with cumulative duration of time leading up to its observed time

    decay_matrix = np.exp(-t_diff * deg_rate) # Decay_matrix[i, m] = decay factor for protein made from RNA available at t_i in cell m
    mask = (t_diff >= 0) # Protein abundance at time t_m can't come from RNA at time t_i > t_m (cell's observed time)
    mask = np.broadcast_to(mask, decay_matrix.shape)
    decay_matrix = np.where(mask, decay_matrix, 0) 
    
    dt = np.diff(t, prepend=t[-1])
    X_bw_dt = X_bw * dt # Multiply each timepoint's RNA by its corresponding time step size (Riemann approximation) 
    X_bw_dt[X_bw_dt < 0] = 0 # X_bw was populated with '-1' for time points after the cell was observed

    p_new = (decay_matrix * X_bw_dt.T).sum(axis=0) # Integrate RNA counts still surviving up to each time point (until the cell was observed)
    P = p_old + p_new # Protein abundance in each cell = pre-existing protein + newly synthesized protein
           
    return P

In [ ]:
n_genes = df_subset.shape[0]
n_cells = X_bw.shape[0]

P = np.full((n_cells, n_genes), -1)

for idx, gene in enumerate(df_subset['Gene']):
    file_path = f"/mnt/lareaulab/reliscu/projects/Chronocell/analyses/janssens_2025_preprint/data/RNA_history_per_gene/eLNPs_var>1.5_208_genes_{gene}_S_RNA_history.pkl"

    with open(file_path, "rb") as f:
        X_bw = pickle.load(f)
    
    deg_rate = median_deg[idx]    
    
    P[:, idx] = impute_protein(X_bw, Q, t, deg_rate, transl_rate=1)

In [274]:
P.shape

(21325, 11)

In [ ]:
P_df = pd.DataFrame(P, columns=df_subset['Gene'].tolist())
P_df.head()


,Bst2,Cd63,Cd9,Ilk,Itgb1,Lamp1,Lgals3,Procr,Sbno1,Sbno2,Vcam1
0,10,120,45,0,85,95,712,0,0,0,0
1,9,117,50,0,84,94,722,2,0,0,0
2,11,123,46,0,85,97,709,0,0,0,0
3,12,124,0,0,85,98,706,0,0,0,0
4,10,119,48,0,84,95,717,0,0,0,0


In [278]:
P_df.to_csv("data/eLNPs_var>1.5_208_genes_RNA_history_imputed_protein.csv", index=False)